### Populate data with doc_ids

In [ ]:
import requests 

docs_url = 'https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-intro/documents.json?raw=1'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

[{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.", 'section': 'General course-related questions', 'question': 'Course - When will the course start?', 'course': 'data-engineering-zoomcamp'}, {'text': 'GitHub - DataTalksClub data-engineering-zoomcamp#prerequisites', 'section': 'General course-related questions', 'question': 'Course - What are the prerequisites for this course?', 'course': 'data-engineering-zoomcamp'}, {'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in

In [5]:
documents

[{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
  'section': 'General course-related questions',
  'question': 'Course - When will the course start?',
  'course': 'data-engineering-zoomcamp'},
 {'text': 'GitHub - DataTalksClub data-engineering-zoomcamp#prerequisites',
  'section': 'General course-related questions',
  'question': 'Course - What are the prerequisites for this course?',
  'course': 'data-engineering-zoomcamp'},
 {'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines 

In [8]:
# rewrite to list comprehension
courses = set([doc['course'] for doc in documents])

courses

{'data-engineering-zoomcamp', 'machine-learning-zoomcamp', 'mlops-zoomcamp'}

In [32]:
# import hashlib

# def generate_document_id(doc):
#     # combined = f"{doc['course']}-{doc['question']}"
#     combined = f"{doc['course']}-{doc['question']}-{doc['text'][:10]}"
#     hash_object = hashlib.md5(combined.encode())
#     hash_hex = hash_object.hexdigest()
#     document_id = hash_hex[:8]
#     return document_id

# for doc in documents:
#     doc['id'] = generate_document_id(doc)


from rag.data.loader import DocumentLoader

loader = DocumentLoader()

for doc in documents:
    id_base = doc['question'] + ' ' + doc['text'] + ' ' + doc['section']
    # Generate and add unique ID
    doc_id = loader.generate_id(id_base)
    doc["doc_id"] = doc_id

In [33]:
documents[3]

{'text': "You don't need it. You're accepted. You can also just start learning and submitting homework without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.",
 'section': 'General course-related questions',
 'question': 'Course - I have registered for the Data Engineering Bootcamp. When can I expect to receive the confirmation email?',
 'course': 'data-engineering-zoomcamp',
 'doc_id': '3292c0188f9fe1582916d8b6db39659f'}

In [34]:
from collections import defaultdict

In [35]:
hashes = defaultdict(list)

for doc in documents:
    doc_id = doc['doc_id']
    hashes[doc_id].append(doc)

In [36]:
len(hashes), len(documents)

(948, 948)

In [37]:
for k, values in hashes.items():
    if len(values) > 1:
        print(k, len(values))

In [38]:
for doc in documents:
    if doc['doc_id'] == '':
        print(doc)
        print(doc['question'] + ' ' + doc['text'])

In [39]:
import json

In [40]:
with open('documents-with-ids.json', 'wt') as f_out:
    json.dump(documents, f_out, indent=2)

In [41]:
!head documents-with-ids.json

[
  {
    "text": "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  \u201cOffice Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon\u2019t forget to register in DataTalks.Club's Slack and join the channel.",
    "section": "General course-related questions",
    "question": "Course - When will the course start?",
    "course": "data-engineering-zoomcamp",
    "doc_id": "12062254f79edd2c2aa8b975638e42b9"
  },
  {
    "text": "GitHub - DataTalksClub data-engineering-zoomcamp#prerequisites",


### Generate questions using connected with doc_ids using OpenAI API

In [42]:
prompt_template = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record. 

The record:

section: {section}
question: {question}
answer: {text}

Provide the output in parsable JSON without using code blocks:

["question1", "question2", ..., "question5"]
""".strip()

In [43]:
from openai import OpenAI
from rag.llm.openai_client import OpenAIClient

custom_client = OpenAIClient(load_env=True)
key = custom_client.api_key

client = OpenAI()
client = OpenAI(api_key=key)


In [44]:
def generate_questions(doc):
    prompt = prompt_template.format(**doc)

    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[{"role": "user", "content": prompt}]
    )

    json_response = response.choices[0].message.content
    return json_response

In [45]:
from tqdm.auto import tqdm

In [46]:
results = {}

In [47]:
for doc in tqdm(documents): 
    doc_id = doc['doc_id']
    if doc_id in results:
        continue

    questions = generate_questions(doc)
    results[doc_id] = questions

  0%|          | 0/948 [00:00<?, ?it/s]

In [52]:
import pickle

In [53]:
# Write to binary file
with open('results.pkl', 'wb') as f_out:
    pickle.dump(results, f_out)

In [68]:
with open('results.pkl', 'rb') as f_in:
    results = pickle.load(f_in)

In [75]:
results

{'12062254f79edd2c2aa8b975638e42b9': '["When exactly does the course commence?", "How do I join the course announcements?", "What time will the first live session occur?", "Where can I register before the course begins?", "How do I subscribe to the course calendar?"]',
 'd6b01635228edd5fef55516a1bc0565e': '[\n    "What prior knowledge or skills should I have before enrolling in the course?",\n    "Is there anything specific I need to know or prepare before starting this class?",\n    "Before taking this course, what foundational information is necessary?",\n    "What are the required qualifications to participate in this program?",\n    "What should I be familiar with prior to beginning the course?"\n]',
 '06168b38a17abd05e2dfcbc1fd77653a': '[\n    "Is late registration possible in the course?",\n    "Am I allowed to submit homeworks after the course begins?",\n    "Can I participate if I miss the official start date?",\n    "Will I face deadlines for submitting final projects?",\n    

In [82]:
import json

parsed_results = {}

for doc_id, json_questions in results.items():
    parsed_results[doc_id] = json.loads(json_questions)

In [84]:
parsed_results, len(parsed_results)

({'12062254f79edd2c2aa8b975638e42b9': ['When exactly does the course commence?',
   'How do I join the course announcements?',
   'What time will the first live session occur?',
   'Where can I register before the course begins?',
   'How do I subscribe to the course calendar?'],
  'd6b01635228edd5fef55516a1bc0565e': ['What prior knowledge or skills should I have before enrolling in the course?',
   'Is there anything specific I need to know or prepare before starting this class?',
   'Before taking this course, what foundational information is necessary?',
   'What are the required qualifications to participate in this program?',
   'What should I be familiar with prior to beginning the course?'],
  '06168b38a17abd05e2dfcbc1fd77653a': ['Is late registration possible in the course?',
   'Am I allowed to submit homeworks after the course begins?',
   'Can I participate if I miss the official start date?',
   'Will I face deadlines for submitting final projects?',
   "Should I be concern

In [85]:
doc_index = {d['doc_id']: d for d in documents}

doc_index

{'12062254f79edd2c2aa8b975638e42b9': {'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
  'section': 'General course-related questions',
  'question': 'Course - When will the course start?',
  'course': 'data-engineering-zoomcamp',
  'doc_id': '12062254f79edd2c2aa8b975638e42b9'},
 'd6b01635228edd5fef55516a1bc0565e': {'text': 'GitHub - DataTalksClub data-engineering-zoomcamp#prerequisites',
  'section': 'General course-related questions',
  'question': 'Course - What are the prerequisites for this course?',
  'course': 'data-engineering-zoomcamp',
  'doc_id': 'd6b01635

In [87]:
final_results = []

for doc_id, questions in parsed_results.items():
    course = doc_index[doc_id]['course']
    for q in questions:
        final_results.append((q, course, doc_id))

In [88]:
final_results

[('When exactly does the course commence?',
  'data-engineering-zoomcamp',
  '12062254f79edd2c2aa8b975638e42b9'),
 ('How do I join the course announcements?',
  'data-engineering-zoomcamp',
  '12062254f79edd2c2aa8b975638e42b9'),
 ('What time will the first live session occur?',
  'data-engineering-zoomcamp',
  '12062254f79edd2c2aa8b975638e42b9'),
 ('Where can I register before the course begins?',
  'data-engineering-zoomcamp',
  '12062254f79edd2c2aa8b975638e42b9'),
 ('How do I subscribe to the course calendar?',
  'data-engineering-zoomcamp',
  '12062254f79edd2c2aa8b975638e42b9'),
 ('What prior knowledge or skills should I have before enrolling in the course?',
  'data-engineering-zoomcamp',
  'd6b01635228edd5fef55516a1bc0565e'),
 ('Is there anything specific I need to know or prepare before starting this class?',
  'data-engineering-zoomcamp',
  'd6b01635228edd5fef55516a1bc0565e'),
 ('Before taking this course, what foundational information is necessary?',
  'data-engineering-zoomcam

### Prepare ground truth data file

In [90]:
import pandas as pd

In [91]:
df = pd.DataFrame(final_results, columns=['question', 'course', 'document'])

In [92]:
df

,question,course,document
0,When exactly does the course commence?,data-engineering-zoomcamp,12062254f79edd2c2aa8b975638e42b9
1,How do I join the course announcements?,data-engineering-zoomcamp,12062254f79edd2c2aa8b975638e42b9
2,What time will the first live session occur?,data-engineering-zoomcamp,12062254f79edd2c2aa8b975638e42b9
3,Where can I register before the course begins?,data-engineering-zoomcamp,12062254f79edd2c2aa8b975638e42b9
4,How do I subscribe to the course calendar?,data-engineering-zoomcamp,12062254f79edd2c2aa8b975638e42b9
...,...,...,...
4703,How can I remove AWS infrastructure created us...,mlops-zoomcamp,958672bf304e1660d019c2368319cdff
4704,What is the procedure for destroying infrastru...,mlops-zoomcamp,958672bf304e1660d019c2368319cdff
4705,What commands are needed to delete infrastruct...,mlops-zoomcamp,958672bf304e1660d019c2368319cdff
4706,How do I reconfigure before destroying infrast...,mlops-zoomcamp,958672bf304e1660d019c2368319cdff


In [93]:
df.to_csv('ground-truth-data.csv', index=False)

In [94]:
!head ground-truth-data.csv

question,course,document
When exactly does the course commence?,data-engineering-zoomcamp,12062254f79edd2c2aa8b975638e42b9
How do I join the course announcements?,data-engineering-zoomcamp,12062254f79edd2c2aa8b975638e42b9
What time will the first live session occur?,data-engineering-zoomcamp,12062254f79edd2c2aa8b975638e42b9
Where can I register before the course begins?,data-engineering-zoomcamp,12062254f79edd2c2aa8b975638e42b9
How do I subscribe to the course calendar?,data-engineering-zoomcamp,12062254f79edd2c2aa8b975638e42b9
What prior knowledge or skills should I have before enrolling in the course?,data-engineering-zoomcamp,d6b01635228edd5fef55516a1bc0565e
Is there anything specific I need to know or prepare before starting this class?,data-engineering-zoomcamp,d6b01635228edd5fef55516a1bc0565e
"Before taking this course, what foundational information is necessary?",data-engineering-zoomcamp,d6b01635228edd5fef55516a1bc0565e
What are the required qualifications to participate in thi